# Lentils foreign-object segmentation with DynUNet — training

Trains the `cuvis-ai-unet` plugin's **DynUNet** on the public
[`cubert-gmbh/XMR_Industrial_Foreign_Object_Detection_Lentils`](https://huggingface.co/datasets/cubert-gmbh/XMR_Industrial_Foreign_Object_Detection_Lentils)
dataset in the plugin family's two-phase pattern:

1. **Phase 1 — statistics**: `StatisticalTrainer` fits the z-score normalizer on full frames.
2. **Phase 2 — gradients**: `GradientTrainer` trains DynUNet against `DiceLoss + CrossEntropyLoss`
   on foreground-biased 128 px crops (nnU-Net-style oversampling of the ~0.06 % foreground).

The pipeline graph is built **in code** by the shared engine
([`examples/lentils/_engine.py`](../../examples/lentils/_engine.py)):
`DataSource → Norm → Augment → DynUNet → {DiceLoss, CrossEntropyLoss}` — the normalizer sits
upstream of augmentation so its statistics describe full frames both when fitted and applied.
The trained result is saved with `pipeline.save_to_file` (YAML + weights) and restored in the
inference notebook.

**Prerequisites** (see [`HF_DATASET.md`](HF_DATASET.md) and the repo README's prerequisites
matrix): `uv sync --extra notebooks` from the repo root, the `cuvis-ai-augment` plugin, a
cuvis-ai with the running-stats `ZScoreNormalizer` (or set `NORMALIZER = "persample"` below),
and — for the default HuggingFace data path — the Cuvis C++ SDK with a matching `cuvis` pin.
A GPU is strongly recommended.

This notebook runs a **tutorial-sized** configuration (a few frames, a small net, a few
epochs) so it completes in minutes; the closing cell shows the champion invocation
(fg-IoU ≈ 0.79) via the `train.py` CLI.

In [ ]:
from pathlib import Path

import utils

cfg = utils.resolve_config()
print(f"data source     : {cfg['data_source']}")
print(f"unet manifest   : {cfg['unet_manifest']}")
print(f"augment manifest: {cfg['augment_manifest']}")

## Tutorial knobs

`LIMIT` caps frames per split (the full split is 808/148/180), `REPEAT` is the
patches-per-frame multiplicity (the champion used 4), and the small `FEATURES` keep the
tutorial fast — the champion topology is `(32, 64, 128, 256, 512)`.

In [ ]:
LIMIT = 12          # frames per split (0 = all)
REPEAT = 2          # train-row multiplicity (champion: 4)
EPOCHS = 3          # champion: 20
FEATURES = (16, 32, 64)  # champion: (32, 64, 128, 256, 512)
PATCH = 128
BATCH = 4
NUM_WORKERS = 4
NORMALIZER = "zscore"    # "persample" if your cuvis-ai predates running-stats support
OUT_DIR = Path("outputs/unet2d")

## Data

`hf` mode downloads the `.cu3s` sessions + COCO polygons and converts the needed frames to
per-frame NPZ (cube + rasterized mask); `local` mode reuses a prepared CSV
(`LENTILS_SPLITS_CSV`). Either way the result is a `(split, npz_path, image_id)` CSV with the
train rows repeated `REPEAT` times.

In [ ]:
if cfg["data_source"] == "hf":
    splits_csv = utils.ensure_lentils_npz(cfg["npz_out"], limit=LIMIT, repeat=REPEAT)
else:
    base = utils.resolve_splits_csv() if hasattr(utils, "resolve_splits_csv") else cfg["splits_csv"]
    work = Path("outputs"); work.mkdir(parents=True, exist_ok=True)
    small = utils.subsample_splits_csv(base, LIMIT, work / "tutorial_splits.csv") if LIMIT else base
    splits_csv = utils.repeat_train_rows(small, work / f"tutorial_splits_x{REPEAT}.csv", REPEAT)
print("splits csv:", splits_csv)

## Build the pipeline and train

`register_plugins` loads the unet + augment manifests into a `NodeRegistry` (needed again at
restore time), `build_graph` wires the six nodes, and `train` runs both phases and saves the
artifact (`pipeline.yaml` + `pipeline.pt` + `run.json`).

In [ ]:
eng = utils.import_engine()

eng.register_plugins(cfg["unet_manifest"], cfg["augment_manifest"])
pipe = eng.build_graph(
    mode="2d",
    features=FEATURES,
    patch=PATCH,
    normalizer=NORMALIZER,
    tile_batch=16,
)
print("pipeline nodes:", [n.name for n in pipe.nodes])

In [ ]:
dm = eng.make_datamodule(splits_csv, batch_size=BATCH, num_workers=NUM_WORKERS)
artifact = eng.train(
    pipe,
    dm,
    epochs=EPOCHS,
    out_dir=OUT_DIR,
    run_meta={"tutorial": True, "limit": LIMIT, "repeat": REPEAT},
)
print("artifact:", artifact)

## Reproducing the champion

The published numbers (fg-IoU **0.79** / fg-Dice **0.88** / image-AUROC **0.998**) come from the
full split with the deep topology and 20 epochs — the same engine, driven by the CLI:

```bash
python examples/lentils/train.py \
    --splits-csv <full-split-csv-with-repeat-4> \
    --epochs 20 --batch 8 --num-workers 4 --out runs/2d128
python examples/lentils/evaluate.py --pipeline runs/2d128/pipeline.yaml \
    --splits-csv <full-split-csv>
```

Continue with [`02_inference.ipynb`](02_inference.ipynb) to evaluate and visualize the artifact
this notebook just saved.